In [8]:
import sys
sys.path.append("../src")
import numpy as np
import mlflow as ml
import _conditions1 as cn
import _sim_iter1 as si
import _loggers1 as lo
import _param1 as prm
import time


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)




In [90]:
EXP_NAME = "simple_test_cov"
RUN_NAME = "simple_1"

ITERS = 1
SEED_ADDL=99
N = 400

PLOT_ALL=True

simple_2_2 = {
    "type": "Simple, 2 group, 50% cens",
    "x_vars": 1, 
    "VAR_CLASS": [2],
    "VAR_PROB":[.5],
    "scale_f": "12 - .5 *x_mat[:,0]",
    "shape_f": "1.2 + 1 *x_mat[:,0]",
    # "cens_scale":5.9
	"cens_scale":1.1
}

# Comp2
SPLIT_RULES2 =  [
    "pmb.ContinuousSplitRule()", # time
    "pmb.ContinuousSplitRule()" # time
]
MODEL_DICT = {"trees": 50,
	"split_rules": SPLIT_RULES2
}
SAMPLER_DICT = {
	"draws": 2000,
	"tune": 1000,
	"cores": 4,
	"chains": 4,
	"compute_convergence_checks": False,
    "k":1,
    "update_fraction":.5,
    "step_size":.3,
    "thin":10
}

In [11]:
import _functions1 as fn
scenario, x_mat, event_dict, sv_true, sv_scale_true = fn.get_sim(np.random.default_rng(99), 400, **simple_2_2)

mean shape 1.7399999999999993
mean scale 11.73
mean time draws 9.94368442441474
cens mean 13.065


In [13]:
event_dict

{'t_event': array([[ 2.],
        [ 2.],
        [ 3.],
        [ 9.],
        [ 7.],
        [13.],
        [ 1.],
        [13.],
        [ 2.],
        [ 2.],
        [ 3.],
        [ 9.],
        [ 4.],
        [ 3.],
        [ 6.],
        [ 4.],
        [12.],
        [14.],
        [ 6.],
        [ 9.],
        [ 1.],
        [ 2.],
        [14.],
        [ 6.],
        [ 1.],
        [10.],
        [14.],
        [ 5.],
        [23.],
        [ 6.],
        [12.],
        [ 6.],
        [ 4.],
        [ 7.],
        [ 6.],
        [ 2.],
        [ 4.],
        [ 7.],
        [ 5.],
        [17.],
        [ 6.],
        [ 6.],
        [ 7.],
        [ 5.],
        [ 7.],
        [ 7.],
        [ 7.],
        [12.],
        [ 4.],
        [ 6.],
        [ 6.],
        [ 2.],
        [ 6.],
        [16.],
        [ 1.],
        [14.],
        [13.],
        [ 3.],
        [ 7.],
        [ 1.],
        [17.],
        [ 3.],
        [ 4.],
        [ 5.],
        [13.],
        [ 2.],

In [14]:
sv_true_mean = sv_true["sv_true"].mean(0)
print(sv_true_mean)
qnt_t = fn.get_quant_times(sv_true_c=sv_true_mean, sv_true= sv_true).flatten()
print(qnt_t)
event_dict = fn.get_quant_events(qnt_t=qnt_t, event=event_dict)


[0.97476116 0.93803617 0.89323465 0.84162095 0.78438479 0.72286393
 0.65855675 0.59306112 0.52798117 0.46482565 0.40491461 0.34930694
 0.29875636 0.25369842 0.21426658 0.18033191 0.15155833 0.12746504
 0.10748815 0.09103531 0.07752938 0.06643938 0.0572991  0.04971492
 0.04336532 0.03799494 0.03340536]
[ 3  6  9 14 19]


In [18]:
x_tst, x_tst_idx = np.unique(x_mat, return_index=True)
cens_perc = event_dict["status"][event_dict["status"] == 0].shape[0]/event_dict["status"].shape[0]
print(x_tst)
print(x_tst_idx)

[0. 1.]
[6 0]


In [19]:
# get uniq times and quantiles as indexes
uniq_t = np.unique(event_dict["t_event"]) 
print(uniq_t)

# qnt_t, qnt_idx = get_quant_times(uniq_t, uniq = False)
true_t = sv_true["true_times"]
true_t


[ 3.  6.  9. 14. 19.]


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27])

In [32]:
sv_true["sv_true"][x_tst_idx[0]]
true_t[qnt_t.astype("int")-1]
# qnt_t.astype("int")-1

array([ 3,  6,  9, 14, 19])

In [ ]:

# this could be the source the index minus 1    # singular sv true
sv_true_r0 = sv_true["sv_true"][int(x_tst_idx[0]),:]
sv_true_r0 = sv_true_r0[qnt_t.astype("int") - 1]
sv_true_r1 = sv_true["sv_true"][int(x_tst_idx[1]),:]
sv_true_r1 = sv_true_r1[qnt_t.astype("int") - 1]

In [22]:
sv_true_r0
sv_true_r1

array([0.9493143 , 0.78741506, 0.55812398, 0.21405757, 0.04889775])

In [ ]:
import bart_surv_adj as sb
def pb_sb_sub(sv_prob, msk):
    sv_m = sv_prob["sv"][:,msk,:].mean(0)
    sv_q = np.quantile(sv_prob["sv"][:,msk,:], [0.025,0.975], axis=0)    
    return sv_m, sv_q

def get_hdi_ci_2(pb_sv):
    s1 = pb_sv["sv"].shape[0]
    s2 = pb_sv["sv"].shape[1]
    s3 = pb_sv["sv"].shape[2]
    hdi_pre = az.convert_to_dataset(pb_sv["sv"].reshape(1,s1,s2,s3))
    #low-high
    hdi_lh = az.hdi(hdi_pre, prob=.95).x.values
    # low-high
    ci_lh = np.quantile(pb_sv["sv"], [0.025, 0.975], 0)
    return hdi_lh, ci_lh

def get_py_bart_surv2(x_mat, event_dict, model_dict, sampler_dict):
    trn = sb.get_surv_pre_train(y_time = event_dict["t_event"], y_status=event_dict["status"], x = x_mat, weight=None)
    return trn
    post_test = sb.get_posterior_test(y_status=event_dict["status"], y_time=event_dict["t_event"], x=x_mat)
    # return post_test
    # y_sk = sb.get_y_sklearn(status = event_dict["status"], t_event=event_dict["t_event"])
    # trn = sb.get_surv_pre_train(y_sk = y_sk, x_sk = x_mat, weight=None)
    # post_test = sb.get_posterior_test(y_sk = y_sk, x_test = x_mat)
    small_post_x = np.unique(post_test["post_x"][:,0]).reshape(-1,1)
    xs = np.hstack([np.repeat(0, small_post_x.shape[0]), np.repeat(1, small_post_x.shape[0])])
    small_coords = np.hstack([np.repeat(0, small_post_x.shape[0]), np.repeat(1, small_post_x.shape[0]), ])
    small_post_x = np.vstack([small_post_x, small_post_x])
    small_post_x = np.hstack([small_post_x, xs.reshape(-1,1)])
    # print(small_post_x)
    # print(small_coords)

    # return trn, post_test, small_post_x, small_coords
    BSM = sb.BartSurvModel(model_config=model_dict, sampler_config=sampler_dict)
    # fit with just the time column
    BSM.fit(y=trn["y"], X=trn["x"], weights=trn["w"], coords = trn["coord"], random_seed=99)
    # post1 = BSM.sample_posterior_predictive(X_pred=small_post_x, coords=small_coords)
    post1 = BSM.sample_posterior_predictive(post_test["post_x"], coords=post_test["coords"])
    # sv_prob = sb.get_sv_prob(post1, small_coords)
    sv_prob = sb.get_sv_prob(post1, post_test["coords"])
    return sv_prob

    sv_1 = pb_sb_sub(sv_prob, 0)
    sv_2 = pb_sb_sub(sv_prob, 1)
    uniq_t = BSM.uniq_times
    hdi_lh, ci_lh = get_hdi_ci_2(sv_prob)
    hdi_1 = hdi_lh[0].T
    hdi_2 = hdi_lh[1].T
    
    # del BSM
    # childs = mp.active_children()
    # for child in childs:
    #     child.kill()
    return (sv_1[0], sv_1[1], hdi_1), (sv_2[0], sv_2[1], hdi_2), uniq_t


In [91]:
import arviz as az
pb_sv = get_py_bart_surv2(x_mat=x_mat, event_dict=event_dict, model_dict=MODEL_DICT, sampler_dict=SAMPLER_DICT)

k 1
new Model


Multiprocess sampling (4 chains in 4 jobs)
FastSurvPGBART: [trees]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 14 seconds.


Starting predict layout...
Processing: Dropping 2000 rows down 800 thinned historical forests (Thin factor: 10)...


In [92]:
pb_sv

{'prob': array([[[0.06176438, 0.2986332 , 0.2986332 , 0.48996119, 0.47684446],
         [0.06176438, 0.2986332 , 0.2986332 , 0.48996119, 0.47684446],
         [0.06176438, 0.2986332 , 0.2986332 , 0.48996119, 0.47684446],
         ...,
         [0.17027287, 0.29165729, 0.29165729, 0.48190209, 0.52938882],
         [0.06176438, 0.2986332 , 0.2986332 , 0.48996119, 0.47684446],
         [0.06176438, 0.2986332 , 0.2986332 , 0.48996119, 0.47684446]],
 
        [[0.04749874, 0.28612361, 0.28612361, 0.4790421 , 0.49576179],
         [0.04749874, 0.28612361, 0.28612361, 0.4790421 , 0.49576179],
         [0.04749874, 0.28612361, 0.28612361, 0.4790421 , 0.49576179],
         ...,
         [0.14170656, 0.23307667, 0.23307667, 0.41427372, 0.43068479],
         [0.04749874, 0.28612361, 0.28612361, 0.4790421 , 0.49576179],
         [0.04749874, 0.28612361, 0.28612361, 0.4790421 , 0.49576179]],
 
        [[0.05383844, 0.30717906, 0.30717906, 0.43337118, 0.48701478],
         [0.05383844, 0.30717906, 0

In [93]:
import matplotlib.pyplot as plt
# plt.hist(pb_sv["sv"].mean(1)[:,0], bins=100)
np.unique(pb_sv["sv"].mean(1)[:,0], return_counts=True)[0].shape

(647,)

In [94]:
qnt_t

array([ 3,  6,  9, 14, 19])

In [95]:
msk = (x_mat==1).flatten()
# pb_sv["sv"][:,msk,:]
print(pb_sv["sv"].mean(0)[msk,:].mean(0))
print(pb_sv["sv"].mean(0)[~msk,:].mean(0))

[0.92066582 0.70375155 0.51798397 0.22399078 0.09688265]
[0.84324746 0.60554099 0.44434829 0.26436707 0.15696958]


In [88]:
msk = (x_mat==1).flatten()
# pb_sv["sv"][:,msk,:]
print(pb_sv["sv"].mean(0)[msk,:].mean(0))
print(pb_sv["sv"].mean(0)[~msk,:].mean(0))


[0.91845996 0.71033388 0.51685231 0.22965508 0.10405746]
[0.84545471 0.60821655 0.4439055  0.26691362 0.1535331 ]


In [ ]:

sv_true_r0_2 = sv_true["sv_true"][int(x_tst_idx[0]),:]
sv_true_r0_2 = sv_true_r0_2[qnt_t.astype("int")]
sv_true_r1_2 = sv_true["sv_true"][int(x_tst_idx[1]),:]
sv_true_r1_2 = sv_true_r1_2[qnt_t.astype("int")]
print(sv_true_r0_2)
print(sv_true_r1_2)

[0.76522871 0.59231233 0.44776186 0.27061735 0.15787623]
[0.90669581 0.71498718 0.47936148 0.16626407 0.03409676]


In [47]:
print(sv_true_r0)
print(sv_true_r1)

[0.82740203 0.64708651 0.49259614 0.30023332 0.17626819]
[0.9493143  0.78741506 0.55812398 0.21405757 0.04889775]


In [45]:
pb_sv

((array([0.83315208, 0.59556217, 0.44823112, 0.27731636, 0.15776741]),
  array([[0.75672766, 0.52235432, 0.36972975, 0.20223054, 0.08045736],
         [0.88900581, 0.67389034, 0.52470423, 0.35544094, 0.2559039 ]]),
  array([[0.76950102, 0.52892598, 0.36974602, 0.19982624, 0.08876584],
         [0.89224654, 0.67500456, 0.52526153, 0.3542971 , 0.26082064]])),
 (array([0.92453066, 0.72058897, 0.51020133, 0.21946898, 0.09005743]),
  array([[0.88750434, 0.65164594, 0.43729551, 0.15748424, 0.04303061],
         [0.96067529, 0.78531267, 0.58503529, 0.3132337 , 0.1555155 ]]),
  array([[0.8915335 , 0.64678444, 0.43583234, 0.14693942, 0.04107133],
         [0.96326229, 0.7729757 , 0.584335  , 0.29564159, 0.14691646]])),
 array([ 3.,  6.,  9., 14., 19.]))

In [96]:
import subprocess
import pandas as pd
def save_to_csv(event_dict, x_mat, file="exp1_tmp.csv"):
    """helper function that passes the simulated event dataset to a csv
    - Used for R-bart

    Args:
        event_dict (_type_): _description_
        x_mat (_type_): _description_
        file (str, optional): _description_. Defaults to "exp1_tmp.csv".
    """
    col = ["t","s"] + ["x"+str(i) for i in range(x_mat.shape[1])]
    df = pd.DataFrame(np.hstack(
        [
            event_dict["t_event"],
            event_dict["status"],
            x_mat
        ]
    ),columns= col)
    exp_name = file
    path = f"../data/{exp_name}"
    with open(path, 'w') as f:
        df.to_csv(f, index=False)
    # df.to_csv(path,index = False)

def get_r_bart2(event_dict, x_mat):
    save_to_csv(event_dict, x_mat, file="exp2_tmp.csv")
    # subprocess.call("/usr/bin/Rscript --vanilla ../R/bart_2.R", shell=True)
    subprocess.call("/usr/local/bin/Rscript --vanilla ../R/bart_2.R", shell=True)
    with open("../data/exp2_tmp_out2.csv","r") as f:
        r_sv = pd.read_csv(f).values
    # r_sv = pd.read_csv("../data/exp2_tmp_out2.csv").values
    msk = r_sv[:,0]==0
    r_sv_m1 = r_sv[msk,1].T
    r_sv_m2 = r_sv[~msk,1].T
    r_sv_q1 = r_sv[msk,2:].T
    r_sv_q2 = r_sv[~msk,2:].T
    return (r_sv_m1, r_sv_q1), (r_sv_m2, r_sv_q2)
    

In [99]:
get_r_bart2(event_dict, x_mat)

Loading required package: nlme
Loading required package: survival


[1] "/Users/jacobtiegs/projects/bart_paper/bart_survival_paper_1000/py"
*****Calling gbart: type=2
*****Data:
data:n,p,np: 945, 2, 10
y1,yn: 0.000000, 1.000000
x1,x[n*p]: 3.000000, 1.000000
xp1,xp[np*p]: 3.000000, 1.000000
*****Number of Trees: 50
*****Number of Cut Points: 4 ... 1
*****burn,nd,thin: 250,1670,10
*****Prior:beta,alpha,tau,nu,lambda,offset: 2,0.95,0.212132,3,1,-0.757622
*****Dirichlet:sparse,theta,omega,a,b,rho,augment: 0,0,1,0.5,1,2,0
*****printevery: 100

MCMC
done 0 (out of 1920)
done 100 (out of 1920)
done 200 (out of 1920)
done 300 (out of 1920)
done 400 (out of 1920)
done 500 (out of 1920)
done 600 (out of 1920)
done 700 (out of 1920)
done 800 (out of 1920)
done 900 (out of 1920)
done 1000 (out of 1920)
done 1100 (out of 1920)
done 1200 (out of 1920)
done 1300 (out of 1920)
done 1400 (out of 1920)
done 1500 (out of 1920)
done 1600 (out of 1920)
done 1700 (out of 1920)
done 1800 (out of 1920)
done 1900 (out of 1920)
time: 2s
trcnt,tecnt: 167,167


((array([0.83245494, 0.59358711, 0.44731984, 0.28742671, 0.17572274]),
  array([[0.7804853 , 0.51815057, 0.36489082, 0.20730756, 0.09653059],
         [0.87997226, 0.66861572, 0.52835064, 0.37520033, 0.26997622]])),
 (array([0.93364492, 0.7232589 , 0.51592325, 0.20618974, 0.08565932]),
  array([[0.89744015, 0.65310059, 0.44158982, 0.1409424 , 0.03136556],
         [0.96095944, 0.78622738, 0.58627387, 0.27861542, 0.15561919]])))

In [ ]:
rpre = pd.read_csv("../data/tmp_output2.csv")
rpre



,Unnamed: 0,t,x1,Unnamed: 3
0,1,3,1,0
1,2,3,1,0
2,3,3,1,0
3,4,3,1,0
4,5,6,1,0
...,...,...,...,...
940,941,3,1,0
941,942,6,1,0
942,943,9,1,0
943,944,14,1,0


In [ ]:
def sim_2s(seed, n, scenario, SPLIT_RULES, model_dict, sampler_dict):
    # set rng as seed given
    if type(seed) is int:
        rng = np.random.default_rng(seed)
    else:
        rng = seed
    # generate survival simulation
    scenario, x_mat, event_dict, sv_true, sv_scale_true = get_sim(rng, n, **scenario)
    
    #get mean survival
    sv_true_mean = sv_true["sv_true"].mean(0)
    qnt_t = get_quant_times(sv_true_c=sv_true_mean, sv_true= sv_true).flatten()
    print(qnt_t)
    event_dict = get_quant_events(qnt_t=qnt_t, event=event_dict)

    
    # get the unique x_test
    x_tst, x_tst_idx = np.unique(x_mat, return_index=True)
    cens_perc = event_dict["status"][event_dict["status"] == 0].shape[0]/event_dict["status"].shape[0]
    
    # get uniq times and quantiles as indexes
    uniq_t = np.unique(event_dict["t_event"]) 
    print(uniq_t)
    
    # qnt_t, qnt_idx = get_quant_times(uniq_t, uniq = False)
    true_t = sv_true["true_times"]
    
    # singular sv true
    sv_true_r0 = sv_true["sv_true"][int(x_tst_idx[0]),:]
    sv_true_r0 = sv_true_r0[qnt_t.astype("int") - 1]
    sv_true_r1 = sv_true["sv_true"][int(x_tst_idx[1]),:]
    sv_true_r1 = sv_true_r1[qnt_t.astype("int") - 1]
    
    # fit kpm    
    ed1, uniq_t1 = ed_sub(event_dict, x_mat, 0)
    ed2, uniq_t2 = ed_sub(event_dict, x_mat, 1)
    k_sv1 = twopop_kpm(ed1, uniq_t)
    k_sv2 = twopop_kpm(ed2, uniq_t)
    
    # print(x_mat)
    # fit bart_py
    pb_sv = get_py_bart_surv2(x_mat=x_mat, event_dict=event_dict, model_dict=MODEL_DICT, sampler_dict=SAMPLER_DICT)

    # fit bart_r
    r_sv = get_r_bart2(event_dict, x_mat)

    return {"cens_perc":cens_perc, 
        "uniq_t":uniq_t, 
        "sv_true_u":(sv_true_r0, sv_true_r1),
        "pb_sv":pb_sv, 
        # "pb_hdi":hdi[0].T, 
        # "pb_ci":ci[:,0,:], 
        "r_sv":r_sv, 
        "k_sv":(k_sv1, k_sv2)
        }


In [ ]:
def iter_simulation_2s(iters, n, seed_addl, scenario, SPLIT_RULES, model_dict, sampler_dict, plot_all=False):
    """_summary_

    Args:
        iters (_type_): _description_
        n (_type_): _description_
        scenario (_type_): _description_
        SPLIT_RULES (_type_): _description_
        model_dict (_type_): _description_
        sampler_dict (_type_): _description_

    Returns:
        _type_: _description_
    """
    # meta_lst = []
    cens_lst = []
    sv_true_lst0 = []
    k_sv_lst0 = []
    k_sv_ci_lst0 = []
    pb_sv_lst0 = []
    pb_sv_ci_lst0 = []
    pb_sv_hdi_lst0 = []
    r_sv_lst0 = []
    r_sv_ci_lst0 = []

    sv_true_lst1 = []
    k_sv_lst1 = []
    k_sv_ci_lst1 = []
    pb_sv_lst1 = []
    pb_sv_ci_lst1 = []
    pb_sv_hdi_lst1 = []
    r_sv_lst1 = []
    r_sv_ci_lst1 = []
    figs = []

    strt = n*seed_addl
    end = strt + iters
    with open("../output_sim2.txt", "w") as f:
        print("new")
    for i in range(strt, end):

        with open("../output_sim2.txt", "a") as f:
            print(f"ITERATION************** {i}", file=f)
        print(f"ITERATION************** {i}")
        odict = fn.sim_2s(i, n, scenario, SPLIT_RULES, model_dict, sampler_dict)
        sv_t_0 = odict["sv_true_u"][0]
        sv_t_1 = odict["sv_true_u"][1]

        k_sv_0 = odict["k_sv"][0][0]
        k_sv_1 = odict["k_sv"][1][0]
        k_sv_ci0 = odict["k_sv"][0][1]
        k_sv_ci1 = odict["k_sv"][1][1]
 
        p_sv_0 = odict["pb_sv"][0][0]
        p_sv_1 = odict["pb_sv"][1][0]
        p_sv_ci0 = odict["pb_sv"][0][1]
        p_sv_ci1 = odict["pb_sv"][1][1]
        p_sv_hdi0 = odict["pb_sv"][0][2]
        p_sv_hdi1 = odict["pb_sv"][1][2]

        r_sv_0 = odict["r_sv"][0][0]
        r_sv_1 = odict["r_sv"][1][0]
        r_sv_ci0 = odict["r_sv"][0][1]
        r_sv_ci1 = odict["r_sv"][1][1]



        # meta_lst.append(meta)
        cens_lst.append(odict["cens_perc"])
        sv_true_lst0.append(sv_t_0)
        k_sv_lst0.append(k_sv_0)
        k_sv_ci_lst0.append(k_sv_ci0)
        pb_sv_lst0.append(p_sv_0)
        pb_sv_ci_lst0.append(p_sv_ci0)
        pb_sv_hdi_lst0.append(p_sv_hdi0)
        r_sv_lst0.append(r_sv_0)
        r_sv_ci_lst0.append(r_sv_ci0)

        sv_true_lst1.append(sv_t_1)
        k_sv_lst1.append(k_sv_1)
        k_sv_ci_lst1.append(k_sv_ci1)
        pb_sv_lst1.append(p_sv_1)
        pb_sv_ci_lst1.append(p_sv_ci1)
        pb_sv_hdi_lst1.append(p_sv_hdi1)
        r_sv_lst1.append(r_sv_1)
        r_sv_ci_lst1.append(r_sv_ci1)


    # if plot_all==False:
        # fig = pltf.plots2(meta, sv_true, k_sv, pb_sv, r_sv)
    fig, ax = plt.subplots(1,2, figsize=(10,10))
    ax[0].step(odict["uniq_t"], odict["sv_true_u"][0], color= "black", where="mid", label = "true")
    ax[1].step(odict["uniq_t"], odict["sv_true_u"][1], color = "black", where= "mid")
    ax[0].step(odict["uniq_t"], odict["k_sv"][0][1][0,:], color = "yellow", where="mid", alpha=0.5, linestyle="dashed", label="k_ci")
    ax[0].step(odict["uniq_t"], odict["k_sv"][0][1][1,:], color = "yellow", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["k_sv"][1][1][0,:], color = "yellow", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["k_sv"][1][1][1,:], color = "yellow", where="mid", alpha=0.5, linestyle="dashed")
    ax[0].step(odict["uniq_t"], odict["r_sv"][0][1][0,:], color = "blue", where="mid", alpha=0.5, linestyle="dashed", label = "r_ci")
    ax[0].step(odict["uniq_t"], odict["r_sv"][0][1][1,:], color = "blue", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["r_sv"][1][1][0,:], color = "blue", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["r_sv"][1][1][1,:], color = "blue", where="mid", alpha=0.5, linestyle="dashed")
    ax[0].step(odict["uniq_t"], odict["pb_sv"][0][1][0,:], color = "red", where="mid", alpha=0.5, linestyle="dashed", label="p_ci")
    ax[0].step(odict["uniq_t"], odict["pb_sv"][0][1][1,:], color = "red", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["pb_sv"][1][1][0,:], color = "red", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["pb_sv"][1][1][1,:], color = "red", where="mid", alpha=0.5, linestyle="dashed")
    ax[0].step(odict["uniq_t"], odict["pb_sv"][0][2][0,:], color = "green", where="mid", alpha=0.5, linestyle="dashed", label="p_hdi")
    ax[0].step(odict["uniq_t"], odict["pb_sv"][0][2][1,:], color = "green", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["pb_sv"][1][2][0,:], color = "green", where="mid", alpha=0.5, linestyle="dashed")
    ax[1].step(odict["uniq_t"], odict["pb_sv"][1][2][1,:], color = "green", where="mid", alpha=0.5, linestyle="dashed")
    ax[0].legend()
    title = f"{scenario['type']} {n}"
    fig.suptitle(title)
    figs.append(fig)
    

    k, p, r = fn.get_metrics2(
        sv_true_lst0, sv_true_lst1,
        k_sv_lst0, k_sv_lst1,
        k_sv_ci_lst0, k_sv_ci_lst1,
        pb_sv_lst0, pb_sv_lst1,
        pb_sv_ci_lst0, pb_sv_ci_lst1,
        pb_sv_hdi_lst0, pb_sv_hdi_lst1,
        r_sv_lst0, r_sv_lst1,
        r_sv_ci_lst0, r_sv_ci_lst1
    )
    # cens = np.array([m[0] for m in meta_lst]).mean()
    cens = np.array(cens_lst).mean(0)
    
    # if plot_all:
        # return meta_lst, (strt,end), cens, k,p,r, figs
    return (strt,end), cens, k,p,r, figs
    # return meta_lst, (strt,end), cens, k,p,r, fig



In [ ]:
seeds, cens,k,p,r,fig = si.iter_simulation_2s(
    iters=ITERS, 
    n=N,
    seed_addl=SEED_ADDL,
    scenario= simple_2_2, 
    SPLIT_RULES=SPLIT_RULES2, 
    model_dict=MODEL_DICT, 
    sampler_dict=SAMPLER_DICT,
    plot_all=PLOT_ALL
)

lo.log_params("2_2", N, cn.simple_2_2, MODEL_DICT, SAMPLER_DICT, seeds)
lo.log_mets("2_2", N, cens, k, p, r)
